In [2]:
"""
Upload processed datasets and model artifacts to AWS S3 securely.

- Credentials are loaded from a .env file or environment variables.
- Automatically logs uploads and handles missing files gracefully.
"""

import os
from pathlib import Path
from dotenv import load_dotenv
import boto3
from botocore.exceptions import NoCredentialsError, PartialCredentialsError

# ---- Load credentials from .env or system environment ----
load_dotenv()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "eu-west-2")
BUCKET_NAME = os.getenv("S3_BUCKET_NAME", "housing-data-regression-cc")

# ---- Project paths ----
PROJECT_ROOT = Path.cwd().parent
local_data_dir = PROJECT_ROOT / "data" / "processed"
local_model_dir = PROJECT_ROOT / "models"

# ---- Initialize S3 client ----
try:
    s3 = boto3.client(
        "s3",
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        region_name=AWS_REGION,
    )
except (NoCredentialsError, PartialCredentialsError):
    print("❌ AWS credentials not found. Check your .env file or environment variables.")
    raise

# ---- Helper function ----
def upload_file(local_path: Path, s3_key: str):
    """Upload a single file to the configured S3 bucket."""
    if not local_path.exists():
        print(f"❌ File not found: {local_path}")
        return

    print(f"⬆️ Uploading {local_path} → s3://{BUCKET_NAME}/{s3_key}")
    try:
        s3.upload_file(str(local_path), BUCKET_NAME, s3_key)
        print(f"✅ Uploaded: {s3_key}")
    except NoCredentialsError:
        print("❌ No AWS credentials found. Aborting upload.")
    except Exception as e:
        print(f"⚠️ Failed to upload {local_path.name}: {e}")

# ---- Upload required datasets ----
upload_file(local_data_dir / "feature_engineered_holdout.csv", "processed/feature_engineered_holdout.csv")
upload_file(local_data_dir / "cleaning_holdout.csv", "processed/cleaning_holdout.csv")
upload_file(local_data_dir / "feature_engineered_train.csv", "processed/feature_engineered_train.csv")

# ---- Upload trained model ----
upload_file(local_model_dir / "xgb_best_model.pkl", "models/xgb_best_model.pkl")

⬆️ Uploading c:\Users\bhavy\Desktop\github-projects\cc-regression-ml\Regression_ML_EndtoEnd\data\processed\feature_engineered_holdout.csv → s3://housing-data-regression-cc/processed/feature_engineered_holdout.csv
✅ Uploaded: processed/feature_engineered_holdout.csv
⬆️ Uploading c:\Users\bhavy\Desktop\github-projects\cc-regression-ml\Regression_ML_EndtoEnd\data\processed\cleaning_holdout.csv → s3://housing-data-regression-cc/processed/cleaning_holdout.csv
✅ Uploaded: processed/cleaning_holdout.csv
⬆️ Uploading c:\Users\bhavy\Desktop\github-projects\cc-regression-ml\Regression_ML_EndtoEnd\data\processed\feature_engineered_train.csv → s3://housing-data-regression-cc/processed/feature_engineered_train.csv
✅ Uploaded: processed/feature_engineered_train.csv
⬆️ Uploading c:\Users\bhavy\Desktop\github-projects\cc-regression-ml\Regression_ML_EndtoEnd\models\xgb_best_model.pkl → s3://housing-data-regression-cc/models/xgb_best_model.pkl
✅ Uploaded: models/xgb_best_model.pkl
